# E13 — truncation decomposition and the completed cospectral tracer

E7 established that QuIC's cycle-count information is rank-stratified: the
head of the sorted probability vector carries the signal, and no target
gains materially past k = 1000. But E7 left two things open. First, the
raw head confounds two quantities — the **shape** of the retained
probabilities and their **cumulative retained mass** — because the head is
not renormalized, so a ridge model can use either. Second, the notebook's
planned cospectral percentile tracer never produced outputs, so E7
supports no claim about how truncation changes the relative standing of
cospectral separations. E13 resolves both.

**Four head representations, per k.** For each k in {25, 50, 100, 200,
400, 1000}:
1. **raw head** h_k = (p_(1), ..., p_(k)) — the E7 representation;
2. **normalized head** h_k / sum(h_k) — shape only, mass divided out;
3. **retained mass only** m_k = sum_{i<=k} p_(i) — a single scalar;
4. **normalized head + mass** [h_k / sum(h_k), m_k] — shape and mass as
   separate coordinates.

Comparing (1) against (2), (3), and (4) decomposes the head's predictive
value into shape and mass. If normalized-head performance matches raw,
the shape carries it; if mass-only already suffices, concentration is the
whole story; if (4) beats both (2) and (3), shape and mass are
complementary.

**Targets.** C3, C4, C5, C6, diamonds, girth, and log2|Aut| within
cospectral groups. The cycle and diamond targets are the E7/E2R panel; the
within-group |Aut| target ties the decomposition to the E8B/E12 functional
result — it asks whether the symmetry signal that ranks |Aut| lives in the
head's shape, its mass, or both.

**The cospectral tracer, completed.** For every exact cospectral pair
(from the E8A trace-tuple census, recomputed here) the notebook reports,
at each k: the full-vector L1 separation, the raw-head L1 separation, the
normalized-head L1 separation, the retained-mass difference, the fraction
of full separation retained by the raw head, and the pair's percentile
among all count-identical pairs at that k. This is the analysis E7 planned
and did not deliver, and it answers whether cospectral mate separations
concentrate in the head the same way cycle-count signal does, or whether
they are a tail phenomenon.

**Protocol.** Cycle and diamond targets use the standing RidgeCV probe
(alphas = logspace(-14, 2, 17), cv = 5) on the frozen seed-0 outer folds.
The |Aut|-within-groups target reuses the E12 group-preserving ranking
(leave-one-cospectral-group-out, intercept-free antisymmetric logistic,
fixed C = 1.0) so its numbers sit beside E12's on the same footing. Cycle
and diamond targets are computed in-notebook and certified by the tr(A^3),
tr(A^4), and tr(A^6) identities.

**Dry-run finding, on record (n = 14).** The decomposition already shows
the shape-versus-mass split the moment ladder predicts. Triangle count is
carried by mass alone — the mass-only column reaches 0.996 at k = 25,
because C3 is a purity (Sum p^2) quantity. C5 is the opposite: mass-only
stays dead (near 0 at every k) while the normalized head tracks the raw
head to 0.938 at k = 1000, so C5's signal is pure shape, matching its
fine-L-statistic carrier. C4 sits between, and diamonds and girth are
shape-carried and shallow (0.93 and 0.92 by k = 25 in the normalized
head, mass-only weak). The norm+mass column never beats the better of its
two parts by more than fold noise, so shape and mass are not strongly
complementary for these targets. The tracer runs: median fraction of full
L1 retained by the head climbs 0.16 to 0.96 across the depth grid, and the
count-identical percentile drifts rather than moving monotonically,
consistent with E2C's pair-dependent head concentration.

**Runtime.** Cycle and diamond ridges dominate: four representations x six
k x seven targets, but every representation is a k <= 1000 head, so this
is far cheaper than the full-vector probes — roughly 30 minutes at n = 14
and 1.5 to 2 hours at n = 16. The tracer is arithmetic on stored vectors,
seconds. RUN_NS is the session knob; results checkpoint per census.

## Environment

In [1]:
import pickle
import time

from collections import defaultdict
from itertools import combinations

import numpy as np
import networkx as nx

from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from tqdm.notebook import tqdm


from scipy.linalg import LinAlgWarning
import warnings
warnings.filterwarnings('ignore', category=LinAlgWarning)

In [2]:
SEED = 0
RIDGE_ALPHAS = np.logspace(-14, 2, 17)
NUM_FOLDS = 5

CENSUS_BASES = {
    14: "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/",
    16: "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/",
}
EXPECTED_CUBIC_COUNTS = {14: 509, 16: 4060}

TRUNCATION_DEPTHS = (25, 50, 100, 200, 400, 1000)
CYCLE_TARGETS = ('C3', 'C4', 'C5', 'C6', 'diamonds', 'girth')
REPRESENTATION_NAMES = ('raw_head', 'normalized_head', 'mass_only',
                        'normalized_head_plus_mass')

# E8A group-level target for the within-cospectral-group ranking arm.
GROUP_RANKING_C = 1.0

RUN_NS = (14, 16)

## Load censuses, targets (identity-certified), cospectral groups, folds

In [3]:
def compute_targets(adjacency_matrix):
    graph = nx.from_numpy_array(adjacency_matrix)
    cycle_counter = defaultdict(int)
    for cycle_vertices in nx.simple_cycles(graph, length_bound=6):
        cycle_counter[len(cycle_vertices)] += 1
    edge_u, edge_v = np.nonzero(np.triu(adjacency_matrix, k=1))
    common_neighbor_counts = (adjacency_matrix[edge_u]
                              * adjacency_matrix[edge_v]).sum(axis=1)
    return {'C3': cycle_counter[3], 'C4': cycle_counter[4],
            'C5': cycle_counter[5], 'C6': cycle_counter[6],
            'diamonds': int((common_neighbor_counts == 2).sum()),
            'girth': nx.girth(graph),
            'log2_aut': float(np.log2(sum(
                1 for _ in nx.isomorphism.GraphMatcher(
                    graph, graph).isomorphisms_iter())))}


CENSUS = {}
for num_vertices in RUN_NS:
    with open(CENSUS_BASES[num_vertices]
              + f"n{num_vertices}_data_dict.pkl", 'rb') as census_file:
        census_data = pickle.load(census_file)
    graph_keys = sorted(census_data)
    assert len(graph_keys) == EXPECTED_CUBIC_COUNTS[num_vertices]

    adjacency_matrices = np.stack(
        [census_data[graph_key]['adj_mat'] for graph_key in graph_keys]
    ).astype(np.int64)
    quic_matrix = np.vstack(
        [census_data[graph_key]['exact_vector'] for graph_key in graph_keys])
    assert np.all(np.diff(quic_matrix, axis=1) <= 0)
    assert np.abs(quic_matrix.sum(axis=1) - 1.0).max() < 1e-12

    target_arrays = defaultdict(list)
    for adjacency_matrix in tqdm(adjacency_matrices,
                                 desc=f'n={num_vertices} targets'):
        for name, value in compute_targets(adjacency_matrix).items():
            target_arrays[name].append(value)
    target_arrays = {name: np.array(values, dtype=float)
                     for name, values in target_arrays.items()}

    triangle_counts = target_arrays['C3'].astype(np.int64)
    square_counts = target_arrays['C4'].astype(np.int64)
    hexagon_counts = target_arrays['C6'].astype(np.int64)
    diamond_counts = target_arrays['diamonds'].astype(np.int64)
    walk_matrix = adjacency_matrices @ adjacency_matrices @ adjacency_matrices
    assert np.array_equal(np.trace(walk_matrix, axis1=1, axis2=2),
                          6 * triangle_counts)
    walk_matrix = walk_matrix @ adjacency_matrices
    assert np.array_equal(np.trace(walk_matrix, axis1=1, axis2=2),
                          8 * square_counts + 15 * num_vertices)
    walk_matrix = walk_matrix @ adjacency_matrices @ adjacency_matrices
    assert np.array_equal(np.trace(walk_matrix, axis1=1, axis2=2),
                          87 * num_vertices + 6 * triangle_counts
                          + 96 * square_counts
                          + 12 * (hexagon_counts + diamond_counts))

    # cospectral groups via exact integer trace tuples (E8A method)
    trace_tuples = np.zeros((len(graph_keys), num_vertices), dtype=np.int64)
    walk_matrix = adjacency_matrices.copy()
    trace_tuples[:, 0] = np.trace(walk_matrix, axis1=1, axis2=2)
    for power_index in range(1, num_vertices):
        walk_matrix = walk_matrix @ adjacency_matrices
        trace_tuples[:, power_index] = np.trace(walk_matrix, axis1=1, axis2=2)
    groups_by_tuple = defaultdict(list)
    for graph_index in range(len(graph_keys)):
        groups_by_tuple[tuple(trace_tuples[graph_index])].append(graph_index)
    cospectral_groups = sorted(sorted(members)
                               for members in groups_by_tuple.values()
                               if len(members) > 1)

    fold_splits = list(KFold(n_splits=NUM_FOLDS, shuffle=True,
                             random_state=SEED).split(np.arange(len(graph_keys))))
    CENSUS[num_vertices] = {
        'quic_matrix': quic_matrix, 'target_arrays': target_arrays,
        'cospectral_groups': cospectral_groups, 'fold_splits': fold_splits}
    print(f'n={num_vertices}: {len(graph_keys)} graphs, identities exact, '
          f'{len(cospectral_groups)} cospectral groups')

n=14 targets:   0%|          | 0/509 [00:00<?, ?it/s]

n=14: 509 graphs, identities exact, 3 cospectral groups


n=16 targets:   0%|          | 0/4060 [00:00<?, ?it/s]

n=16: 4060 graphs, identities exact, 41 cospectral groups


## Head representations

Each builder takes the full sorted matrix and a depth k and returns the
feature matrix for that representation. Normalization divides the head by
its own retained mass, which is exactly the shape-versus-mass separation
the decomposition needs.

In [4]:
def build_representation(sorted_matrix, depth, representation_name):
    head = sorted_matrix[:, :depth]
    retained_mass = head.sum(axis=1, keepdims=True)
    if representation_name == 'raw_head':
        return head
    if representation_name == 'normalized_head':
        return head / retained_mass
    if representation_name == 'mass_only':
        return retained_mass
    if representation_name == 'normalized_head_plus_mass':
        return np.hstack([head / retained_mass, retained_mass])
    raise ValueError(representation_name)


def ridge_probe_r2(feature_matrix, target_values, fold_splits):
    fold_scores = []
    for train_indices, test_indices in fold_splits:
        model = RidgeCV(alphas=RIDGE_ALPHAS, cv=5)
        model.fit(feature_matrix[train_indices], target_values[train_indices])
        fold_scores.append(r2_score(target_values[test_indices],
                                    model.predict(feature_matrix[test_indices])))
    return float(np.mean(fold_scores)), float(np.std(fold_scores))

## Decomposition grid — cycle and diamond targets

For every target, representation, and depth: the ridge probe R². The four
representations side by side at each depth are the decomposition.

In [5]:
DECOMPOSITION = {}
for num_vertices in RUN_NS:
    census = CENSUS[num_vertices]
    DECOMPOSITION[num_vertices] = {}
    for target_name in CYCLE_TARGETS:
        target_values = census['target_arrays'][target_name]
        DECOMPOSITION[num_vertices][target_name] = {}
        for depth in tqdm(TRUNCATION_DEPTHS,
                          desc=f'n={num_vertices} {target_name}'):
            for representation_name in REPRESENTATION_NAMES:
                feature_matrix = build_representation(
                    census['quic_matrix'], depth, representation_name)
                mean_r2, std_r2 = ridge_probe_r2(
                    feature_matrix, target_values, census['fold_splits'])
                DECOMPOSITION[num_vertices][target_name][
                    (depth, representation_name)] = (mean_r2, std_r2)

    with open('/kaggle/working/e13_truncation_decomposition.pkl', 'wb') \
            as output_file:
        pickle.dump({'decomposition': DECOMPOSITION,
                     'config': {'depths': TRUNCATION_DEPTHS,
                                'representations': REPRESENTATION_NAMES,
                                'targets': CYCLE_TARGETS, 'seed': SEED}},
                    output_file)
    print(f'n={num_vertices}: decomposition checkpointed')

n=14 C3:   0%|          | 0/6 [00:00<?, ?it/s]

n=14 C4:   0%|          | 0/6 [00:00<?, ?it/s]

n=14 C5:   0%|          | 0/6 [00:00<?, ?it/s]

n=14 C6:   0%|          | 0/6 [00:00<?, ?it/s]

n=14 diamonds:   0%|          | 0/6 [00:00<?, ?it/s]

...

n=16 C5:   0%|          | 0/6 [00:00<?, ?it/s]

n=16 C6:   0%|          | 0/6 [00:00<?, ?it/s]

n=16 diamonds:   0%|          | 0/6 [00:00<?, ?it/s]

n=16 girth:   0%|          | 0/6 [00:00<?, ?it/s]

n=16: decomposition checkpointed


## Within-cospectral-group |Aut| ranking, decomposed

The E12 group-preserving ranking run on each head representation at each
depth, so the symmetry result is decomposed into shape and mass exactly
like the cycle targets. Fixed C = 1.0 as in E12.

In [6]:
def group_ranking_correct(difference_features, labels, group_numbers):
    """Leave-one-group-out correct-pair count, intercept-free antisymmetric
    logistic at fixed C (the E12 protocol)."""
    correct = 0
    for held_out in sorted(set(group_numbers.tolist())):
        train_mask = group_numbers != held_out
        scale = np.sqrt((difference_features[train_mask] ** 2).mean())
        if scale == 0:
            continue
        both_features = np.vstack([difference_features[train_mask],
                                   -difference_features[train_mask]]) / scale
        both_labels = np.concatenate([labels[train_mask], -labels[train_mask]])
        model = LogisticRegression(C=GROUP_RANKING_C, fit_intercept=False,
                                   max_iter=5000, random_state=SEED)
        model.fit(both_features, both_labels)
        for position in np.nonzero(~train_mask)[0]:
            decision = float(model.decision_function(
                (difference_features[position] / scale).reshape(1, -1))[0])
            if decision != 0.0 and np.sign(decision) == labels[position]:
                correct += 1
    return correct


GROUP_RANKING = {}
for num_vertices in RUN_NS:
    census = CENSUS[num_vertices]
    aut_values = census['target_arrays']['log2_aut']
    varying_groups = [group for group in census['cospectral_groups']
                      if len({aut_values[g] for g in group}) > 1]
    base_pairs = []
    for group_number, group in enumerate(varying_groups):
        for first_index, second_index in combinations(group, 2):
            gap = aut_values[first_index] - aut_values[second_index]
            if gap != 0:
                base_pairs.append((group_number, first_index, second_index,
                                   float(np.sign(gap))))
    GROUP_RANKING[num_vertices] = {'pair_count': len(base_pairs),
                                   'varying_groups': len(varying_groups),
                                   'by_representation': {}}
    if not base_pairs:
        print(f'n={num_vertices}: no |Aut|-varying cospectral pairs '
              f'(expected at n=14); skipping ranking arm')
        continue

    group_numbers = np.array([p[0] for p in base_pairs])
    labels = np.array([p[3] for p in base_pairs])
    for depth in tqdm(TRUNCATION_DEPTHS, desc=f'n={num_vertices} |Aut| ranking'):
        for representation_name in REPRESENTATION_NAMES:
            representation_lookup = {
                g: build_representation(
                    census['quic_matrix'][g:g+1], depth, representation_name)[0]
                for g in {idx for _, a, b, _ in base_pairs for idx in (a, b)}}
            difference_features = np.vstack([
                representation_lookup[first_index]
                - representation_lookup[second_index]
                for _, first_index, second_index, _ in base_pairs])
            correct = group_ranking_correct(difference_features, labels,
                                             group_numbers)
            GROUP_RANKING[num_vertices]['by_representation'][
                (depth, representation_name)] = correct
    print(f"n={num_vertices}: |Aut| ranking over {len(base_pairs)} pairs "
          f"in {len(varying_groups)} groups done")

n=14 |Aut| ranking:   0%|          | 0/6 [00:00<?, ?it/s]

n=14: |Aut| ranking over 3 pairs in 3 groups done


n=16 |Aut| ranking:   0%|          | 0/6 [00:00<?, ?it/s]

n=16: |Aut| ranking over 14 pairs in 14 groups done


## The completed cospectral tracer

For every exact cospectral pair, at each depth: full-vector L1, raw-head
L1, normalized-head L1, retained-mass difference, fraction of full L1
retained by the head, and the pair's percentile among all count-identical
pairs. The count-identical reference class is all pairs of graphs sharing
the full cycle-count tuple (C3, C4, C5, C6), matching the E2.5 reference
class doctrine.

In [7]:
def full_l1(sorted_matrix, first_index, second_index):
    return float(np.abs(sorted_matrix[first_index]
                        - sorted_matrix[second_index]).sum())


TRACER = {}
for num_vertices in RUN_NS:
    census = CENSUS[num_vertices]
    sorted_matrix = census['quic_matrix']
    target_arrays = census['target_arrays']

    # count-identical reference class: pairs sharing (C3,C4,C5,C6)
    cycle_signature = defaultdict(list)
    for graph_index in range(sorted_matrix.shape[0]):
        signature = tuple(int(target_arrays[name][graph_index])
                          for name in ('C3', 'C4', 'C5', 'C6'))
        cycle_signature[signature].append(graph_index)
    reference_pairs = [(a, b) for members in cycle_signature.values()
                       if len(members) > 1
                       for a, b in combinations(sorted(members), 2)]

    # reference-class full-vector L1 distribution, per depth, for percentiles
    reference_head_l1 = {depth: np.array(
        [float(np.abs(sorted_matrix[a, :depth]
                      - sorted_matrix[b, :depth]).sum())
         for a, b in reference_pairs])
        for depth in TRUNCATION_DEPTHS}

    pair_rows = []
    for group in census['cospectral_groups']:
        for first_index, second_index in combinations(group, 2):
            full_separation = full_l1(sorted_matrix, first_index, second_index)
            per_depth = {}
            for depth in TRUNCATION_DEPTHS:
                first_head = sorted_matrix[first_index, :depth]
                second_head = sorted_matrix[second_index, :depth]
                raw_l1 = float(np.abs(first_head - second_head).sum())
                first_mass = first_head.sum()
                second_mass = second_head.sum()
                normalized_l1 = float(np.abs(first_head / first_mass
                                             - second_head / second_mass).sum())
                head_percentile = float(
                    (reference_head_l1[depth] <= raw_l1).mean() * 100)
                per_depth[depth] = {
                    'raw_head_l1': raw_l1,
                    'normalized_head_l1': normalized_l1,
                    'retained_mass_difference': float(abs(first_mass
                                                          - second_mass)),
                    'fraction_of_full_retained': (raw_l1 / full_separation
                                                  if full_separation > 0
                                                  else float('nan')),
                    'count_identical_percentile': head_percentile}
            pair_rows.append({'first_index': first_index,
                              'second_index': second_index,
                              'full_l1': full_separation,
                              'per_depth': per_depth})
    TRACER[num_vertices] = {'pair_rows': pair_rows,
                            'reference_pair_count': len(reference_pairs)}
    print(f'n={num_vertices}: tracer complete over {len(pair_rows)} '
          f'cospectral pairs vs {len(reference_pairs)} count-identical '
          f'reference pairs')

with open('/kaggle/working/e13_truncation_decomposition.pkl', 'wb') \
        as output_file:
    pickle.dump({'decomposition': DECOMPOSITION, 'group_ranking': GROUP_RANKING,
                 'tracer': TRACER,
                 'config': {'depths': TRUNCATION_DEPTHS,
                            'representations': REPRESENTATION_NAMES,
                            'targets': CYCLE_TARGETS, 'seed': SEED}},
                output_file)
print('full E13 artifact checkpointed')

n=14: tracer complete over 3 cospectral pairs vs 114 count-identical reference pairs
n=16: tracer complete over 43 cospectral pairs vs 8474 count-identical reference pairs
full E13 artifact checkpointed


## Decomposition tables — shape versus mass

For each target the four representations across depth. The raw-versus-
normalized comparison is the shape question; the mass-only column is the
concentration question.

In [8]:
for num_vertices in RUN_NS:
    print(f'\n=== n={num_vertices} — decomposition (R^2 by representation) ===')
    for target_name in CYCLE_TARGETS:
        print(f'\n{target_name}:')
        print(f'{"k":>6} | {"raw":>14} | {"normalized":>14} | '
              f'{"mass_only":>14} | {"norm+mass":>14}')
        for depth in TRUNCATION_DEPTHS:
            cells = DECOMPOSITION[num_vertices][target_name]
            print(f'{depth:>6} | '
                  f'{cells[(depth, "raw_head")][0]:+.3f}±'
                  f'{cells[(depth, "raw_head")][1]:.3f} | '
                  f'{cells[(depth, "normalized_head")][0]:+.3f}±'
                  f'{cells[(depth, "normalized_head")][1]:.3f} | '
                  f'{cells[(depth, "mass_only")][0]:+.3f}±'
                  f'{cells[(depth, "mass_only")][1]:.3f} | '
                  f'{cells[(depth, "normalized_head_plus_mass")][0]:+.3f}±'
                  f'{cells[(depth, "normalized_head_plus_mass")][1]:.3f}')


=== n=14 — decomposition (R^2 by representation) ===

C3:
     k |            raw |     normalized |      mass_only |      norm+mass
    25 | +1.000±0.000 | +0.999±0.000 | +0.996±0.000 | +1.000±0.000
    50 | +1.000±0.000 | +1.000±0.000 | +0.088±0.020 | +1.000±0.000
   100 | +1.000±0.000 | +1.000±0.000 | +0.965±0.004 | +1.000±0.000
   200 | +1.000±0.000 | +1.000±0.000 | +0.946±0.017 | +1.000±0.000
   400 | +1.000±0.000 | +1.000±0.000 | +0.531±0.033 | +1.000±0.000
  1000 | +1.000±0.000 | +1.000±0.000 | +0.916±0.014 | +1.000±0.000

C4:
     k |            raw |     normalized |      mass_only |      norm+mass
    25 | +0.957±0.009 | +0.822±0.026 | -0.008±0.006 | +0.957±0.008
    50 | +0.988±0.004 | +0.988±0.004 | +0.742±0.020 | +0.988±0.004
   100 | +0.993±0.004 | +0.993±0.003 | +0.060±0.041 | +0.994±0.004
   200 | +0.995±0.003 | +0.995±0.004 | +0.009±0.018 | +0.995±0.003
   400 | +0.998±0.002 | +0.998±0.002 | +0.276±0.047 | +0.998±0.002
  1000 | +0.998±0.002 | +0.998±0.001 | -0.007±0.0

## |Aut| ranking decomposition and tracer summary

In [9]:
for num_vertices in RUN_NS:
    ranking = GROUP_RANKING[num_vertices]
    if not ranking['by_representation']:
        continue
    print(f'\n=== n={num_vertices} — |Aut| within-group ranking '
          f'({ranking["pair_count"]} pairs / {ranking["varying_groups"]} '
          f'groups) ===')
    print(f'{"k":>6} | {"raw":>8} | {"normalized":>10} | {"mass_only":>9} | '
          f'{"norm+mass":>9}')
    for depth in TRUNCATION_DEPTHS:
        row = ranking['by_representation']
        print(f'{depth:>6} | {row[(depth, "raw_head")]:>8} | '
              f'{row[(depth, "normalized_head")]:>10} | '
              f'{row[(depth, "mass_only")]:>9} | '
              f'{row[(depth, "normalized_head_plus_mass")]:>9}')

for num_vertices in RUN_NS:
    tracer = TRACER[num_vertices]
    print(f'\n=== n={num_vertices} — cospectral tracer summary '
          f'({len(tracer["pair_rows"])} pairs) ===')
    print(f'{"k":>6} | {"median raw-head L1":>18} | '
          f'{"median frac retained":>20} | {"median pctile":>13}')
    for depth in TRUNCATION_DEPTHS:
        raw_l1_values = [row['per_depth'][depth]['raw_head_l1']
                         for row in tracer['pair_rows']]
        fraction_values = [row['per_depth'][depth]['fraction_of_full_retained']
                           for row in tracer['pair_rows']]
        percentile_values = [
            row['per_depth'][depth]['count_identical_percentile']
            for row in tracer['pair_rows']]
        print(f'{depth:>6} | {np.median(raw_l1_values):18.3e} | '
              f'{np.nanmedian(fraction_values):20.3f} | '
              f'{np.median(percentile_values):13.1f}')


=== n=14 — |Aut| within-group ranking (3 pairs / 3 groups) ===
     k |      raw | normalized | mass_only | norm+mass
    25 |        1 |          1 |         3 |         1
    50 |        1 |          1 |         2 |         1
   100 |        1 |          1 |         1 |         1
   200 |        1 |          1 |         3 |         1
   400 |        1 |          0 |         0 |         0
  1000 |        1 |          0 |         1 |         0

=== n=16 — |Aut| within-group ranking (14 pairs / 14 groups) ===
     k |      raw | normalized | mass_only | norm+mass
    25 |        7 |          7 |         6 |         6
    50 |       11 |         12 |         9 |        12
   100 |       11 |         11 |         9 |        11
   200 |       13 |         13 |        10 |        13
   400 |       12 |         12 |         7 |        12
  1000 |       11 |         11 |         8 |        11

=== n=14 — cospectral tracer summary (3 pairs) ===
     k | median raw-head L1 | median frac retain

## Results

(Written after the run. The reading order: (1) the identity asserts; (2)
the decomposition tables read raw-versus-normalized per target — where
normalized matches raw, the head's shape carries the signal, and where
mass-only already reaches it, concentration is the whole story; the
expected pattern from E2's moment ladder is that triangles (a purity, i.e.
mass, quantity) should be strong in the mass-only column while C5 (fine
L-statistic content) should need the shape; (3) the norm+mass column,
which shows whether shape and mass are complementary or redundant; (4) the
|Aut| ranking decomposition, which localizes the E8B/E12 symmetry signal
into shape or mass and connects the functional result to the geometry; (5)
the tracer, the analysis E7 could not complete — whether cospectral mate
separations concentrate in the head like cycle signal (fraction retained
near 1 at small k) or are a tail phenomenon (fraction small until large
k), and whether their count-identical percentile rises or falls with
truncation, which E2C found is pair-dependent. Verb discipline: the head
"linearly decodes" the target from shape, from mass, or from both; the
tracer "concentrates" or "does not concentrate" mate separations in the
head. No claim about minimum sufficient dimension from a coarse grid.)